# Compute and plot DA analysis increments

Process posterior-minus-prior increments for selected DA experiments without reading a reference experiment.

In [ ]:
# =============================================================================
# TOP-LEVEL PATH PARAMETERS
# Edit these three paths to relocate inputs, the analysis checkout, or outputs.
# =============================================================================
from pathlib import Path
import sys

INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_out")
WORKFLOW_NAME = "analysis_atm_init"

DIAGNOSTIC_DIR = WORK_DIR / "diagnostic"
if not DIAGNOSTIC_DIR.is_dir():
    raise FileNotFoundError(f"Diagnostic work directory not found: {DIAGNOSTIC_DIR}")
if str(DIAGNOSTIC_DIR) not in sys.path:
    sys.path.insert(0, str(DIAGNOSTIC_DIR))

from configs.output_paths import workflow_output_dirs
WORKFLOW_DATA_DIR, WORKFLOW_FIGURE_DIR = workflow_output_dirs(
    WORKFLOW_NAME,
    output_root=DIAGNOSTIC_OUTPUT_ROOT,
)


## Notes: DA analysis increments

This workflow is independent of reference/observation comparisons. It reads only
the experiments listed in `DA_EXPERIMENTS` and computes:

| Metric | Meaning |
|---|---|
| `increment` | Posterior analysis mean minus prior forecast mean |

Available variables are `PS` at the surface and the 3-D variables `U`, `V`,
`T`, `Q`, `CLDLIQ`, and `CLDICE` at a selected pressure/model level. Add a
`level` for every 3-D request and omit it for `PS`. Request-keyed NetCDF caches
are reused unless `force_compute = True`.

For 3-D variables, CROSS_SECTION_REQUESTS derives time-mean vertical sections of signed or absolute analysis increments together with posterior ensemble spread. The latitude axis produces a zonal mean, while longitude produces a meridional mean. Separate request-keyed caches are reused unless force_compute is True.


## Selection

In [ ]:
ANALYSIS_DATA_DIR = WORKFLOW_DATA_DIR / "remapped_ensemble_diagnostics"
CACHE_DIR = WORKFLOW_DATA_DIR / "atmosphere_metrics"
FIGURE_DIR = WORKFLOW_FIGURE_DIR / "analysis_increments"

DA_EXPERIMENTS = ["DART10-S0", "DART20-S0", "DART40-S0"]
TIMESTAMPS = [
    "2011-12-26-00000",
    "2011-12-26-21600",
    "2011-12-26-43200",
    "2011-12-26-64800",
]

INCREMENT_REQUESTS = [
    {"variable": "PS"},
    # Example pressure-level increment:
    # {"variable": "T", "level": 500},
]

LAT_RANGE = (-90, 90)
LON_RANGE = (-180, 180)
force_compute = False
RUN_WORKFLOW = True
MAP_FIGSIZE = (10, 10)

SURFACE_VARIABLES = {"PS"}
THREE_D_VARIABLES = {"U", "V", "T", "Q", "CLDLIQ", "CLDICE"}

MODEL_LABELS = {
    "CAM6-S0": "CAM6-DART (ENS=80)",
    "CTRL10-S0": "CTRL (ENS=10)",
    "DART10-S0": "EAM-DART (ENS=10)",
    "DART20-S0": "EAM-DART (ENS=20)",
    "DART40-S0": "EAM-DART (ENS=40)",
    "DART40INF0p6-S0": "EAM-DART (ENS=40, INF=0.6)",
}

# Vertical DA-increment sections; each request is cached for every timestamp.
CROSS_SECTION_REQUESTS = [
    {"variable": "T", "horizontal_axis": "latitude", "increment_mode": "absolute"},
]


## Imports

In [ ]:
import matplotlib.pyplot as plt
import xarray as xr

from util.atm_initialization_metrics import (
    DartAssimOutput,
    cache_map_metric,
    cache_increment_cross_section,
    increment_cross_section_cache_path,
    map_cache_path,
    plot_cached_maps,
    plot_cached_increment_cross_sections,
)


## 1. Read raw data and process/cache requested metrics

In [ ]:
if RUN_WORKFLOW:
    for request in INCREMENT_REQUESTS:
        variable = request["variable"]
        level = request.get("level")
        if variable not in SURFACE_VARIABLES | THREE_D_VARIABLES:
            raise ValueError(f"Unsupported variable: {variable}")
        if variable in THREE_D_VARIABLES and level is None:
            raise ValueError(f"A level is required for 3-D variable {variable}")
        if variable in SURFACE_VARIABLES and level is not None:
            raise ValueError(f"Surface variable {variable} must not specify a level")

    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    for timestamp in TIMESTAMPS:
        paths = {
            (experiment, request["variable"], request.get("level")):
                map_cache_path(
                    CACHE_DIR, experiment, timestamp, request["variable"],
                    "increment", request.get("level")
                )
            for experiment in DA_EXPERIMENTS
            for request in INCREMENT_REQUESTS
        }
        if not force_compute and all(path.exists() for path in paths.values()):
            print(f"{timestamp}: all increments exist; reusing cache.")
            continue

        print(f"{timestamp}: deriving missing/forced DA increments.")
        outputs = {
            experiment: DartAssimOutput.from_experiment(
                ANALYSIS_DATA_DIR, experiment, timestamp
            )
            for experiment in DA_EXPERIMENTS
        }
        try:
            for (experiment, variable, level), path in paths.items():
                dataset = cache_map_metric(
                    path, outputs[experiment], None, variable, "increment",
                    LAT_RANGE, LON_RANGE, level=level,
                    force_compute=force_compute,
                )
                dataset.close()
        finally:
            for output in outputs.values():
                for dataset in (
                    output.prior_mean, output.prior_sd,
                    output.post_mean, output.post_sd,
                ):
                    dataset.close()


if RUN_WORKFLOW and CROSS_SECTION_REQUESTS:
    for request in CROSS_SECTION_REQUESTS:
        variable = request["variable"]
        horizontal_axis = request.get("horizontal_axis", "latitude")
        increment_mode = request.get("increment_mode", "absolute")
        if variable not in THREE_D_VARIABLES:
            raise ValueError(f"Cross-section variable must be 3-D: {variable}")
        if horizontal_axis not in {"latitude", "longitude"}:
            raise ValueError(f"Unsupported horizontal_axis: {horizontal_axis}")
        if increment_mode not in {"signed", "absolute"}:
            raise ValueError(f"Unsupported increment_mode: {increment_mode}")

    for timestamp in TIMESTAMPS:
        paths = {
            (experiment, request["variable"], request.get("horizontal_axis", "latitude"), request.get("increment_mode", "absolute")): increment_cross_section_cache_path(
                CACHE_DIR, experiment, timestamp, request["variable"],
                request.get("horizontal_axis", "latitude"),
                request.get("increment_mode", "absolute"),
            )
            for experiment in DA_EXPERIMENTS
            for request in CROSS_SECTION_REQUESTS
        }
        if not force_compute and all(path.exists() for path in paths.values()):
            print(f"{timestamp}: all increment sections exist; reusing cache.")
            continue
        outputs = {experiment: DartAssimOutput.from_experiment(ANALYSIS_DATA_DIR, experiment, timestamp) for experiment in DA_EXPERIMENTS}
        try:
            for (experiment, variable, horizontal_axis, increment_mode), path in paths.items():
                dataset = cache_increment_cross_section(
                    path, outputs[experiment], variable, LAT_RANGE, LON_RANGE,
                    horizontal_axis=horizontal_axis,
                    increment_mode=increment_mode,
                    force_compute=force_compute,
                )
                dataset.close()
        finally:
            for output in outputs.values():
                for dataset in (output.prior_mean, output.prior_sd, output.post_mean, output.post_sd):
                    dataset.close()


## 2. Plot cached metrics

In [ ]:
if RUN_WORKFLOW:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    for timestamp in TIMESTAMPS:
        for request in INCREMENT_REQUESTS:
            variable = request["variable"]
            level = request.get("level")
            paths = {
                experiment: map_cache_path(
                    CACHE_DIR, experiment, timestamp, variable, "increment", level
                )
                for experiment in DA_EXPERIMENTS
            }
            missing = [path for path in paths.values() if not path.is_file()]
            if missing:
                raise FileNotFoundError(
                    "Missing increment caches:\n" +
                    "\n".join(str(path) for path in missing)
                )
            cached = {
                experiment: xr.load_dataset(path)
                for experiment, path in paths.items()
            }
            try:
                figure = plot_cached_maps(
                    cached, variable, "increment", LAT_RANGE, LON_RANGE,
                    label_map=MODEL_LABELS,
                    title=f"{variable} Analysis Increment - {timestamp}",
                    figsize=MAP_FIGSIZE,
                )
                level_tag = "surface" if level is None else f"level-{level:g}"
                figure.savefig(
                    FIGURE_DIR / f"{variable}_increment_{level_tag}_{timestamp}.png",
                    dpi=200, bbox_inches="tight",
                )
                plt.close(figure)
            finally:
                for dataset in cached.values():
                    dataset.close()


if RUN_WORKFLOW and CROSS_SECTION_REQUESTS:
    for request in CROSS_SECTION_REQUESTS:
        variable = request["variable"]
        horizontal_axis = request.get("horizontal_axis", "latitude")
        increment_mode = request.get("increment_mode", "absolute")
        time_means = {}
        for experiment in DA_EXPERIMENTS:
            paths = [
                increment_cross_section_cache_path(
                    CACHE_DIR, experiment, timestamp, variable,
                    horizontal_axis, increment_mode,
                )
                for timestamp in TIMESTAMPS
            ]
            missing = [path for path in paths if not path.is_file()]
            if missing:
                raise FileNotFoundError("Missing increment-section caches:\n" + "\n".join(str(path) for path in missing))
            snapshots = [xr.load_dataset(path) for path in paths]
            try:
                time_mean = xr.concat(
                    snapshots, dim=xr.IndexVariable("analysis_time", TIMESTAMPS)
                ).mean("analysis_time", keep_attrs=True)
                time_mean.attrs["analysis_time"] = "time mean: " + ", ".join(TIMESTAMPS)
                time_means[experiment] = time_mean.load()
            finally:
                for dataset in snapshots:
                    dataset.close()
        try:
            figure = plot_cached_increment_cross_sections(
                time_means, variable, increment_mode=increment_mode,
                label_map=MODEL_LABELS,
                title=f"{variable} {increment_mode.title()} Analysis Increment and Posterior Spread",
            )
            figure.savefig(
                FIGURE_DIR / f"{variable}_{increment_mode}_increment_{horizontal_axis}_time-mean.png",
                dpi=200, bbox_inches="tight",
            )
            plt.close(figure)
        finally:
            for dataset in time_means.values():
                dataset.close()
